In [1]:
import os
os.chdir("/home/philbou/projects/def-rfajber/philbou/process_run")
import process_run as pr
import numba as nb
import xarray as xr
ds_data_6h = xr.open_mfdataset("/home/philbou/scratch/isca_data/RT42_sst_0_bucket/run0301/atmos_monthly.nc")
from datetime import datetime
import numpy as np

/home/philbou/miniconda3/envs/pro_env/lib/python3.12/site-packages/xarray/coding/times.py:724: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  dtype = _decode_cf_datetime_dtype(data, units, calendar, self.use_cftime)


In [2]:
ds = xr.open_dataset("/home/philbou/scratch/isca_data/RT42_sst_0_bucket/run0301/p_age_shape_atmos_monthly.nc")

In [3]:
#pr.add_page_shape_to_diag("RT42_sst_0_bucket", 301, file_name="atmos_monthly.nc")

In [4]:
q = ds_data_6h.sphum
qT = ds_data_6h.sphum_age_1
T = (qT/q )/(24*60**2)

In [5]:
dq_conv = ds_data_6h.dt_qg_convection

In [6]:
t0 = datetime.now()
page = pr.age_precip(T.values, dq_conv.values)
t1 = datetime.now()
print((t1-t0))

/lustre06/project/6084782/philbou/process_run/process_run.py:30: RuntimeWarning: invalid value encountered in divide
  updated_mu = (cur_age * cur_dq + dq_l * cur_mu) / cur_dq_l


0:00:40.263075


In [7]:
def age_precip(tau, dq):
    # tau, dq: [time, plev, lat, lon]
    _, plev, _, _ = tau.shape

    cur_mu = np.zeros_like(tau[:, 0, :, :])
    dq_l  = dq[:, 0, :, :].copy()

    for i in range(1, plev):
        cur_age = tau[:, i, :, :]
        cur_dq  = dq[:, i, :, :]
        cur_dq_l = cur_dq + dq_l

        # mask where precipitation occurs
        mask = cur_dq < 0.0

        # Only compute updated values for masked points
        updated_mu = (cur_age * cur_dq + dq_l * cur_mu) / cur_dq_l

        # In-place update — NO np.where
        cur_mu[mask] = updated_mu[mask]

        # update dq_l: zero if cur_dq_l > 0 else keep cur_dq_l
        mask_pos = cur_dq_l > 0
        dq_l[mask_pos] = 0.0
        dq_l[~mask_pos] = cur_dq_l[~mask_pos]

    return cur_mu

In [8]:
def age_precip_column_numba(tau, dq):
    plev = tau.shape[0]
    cur_mu = 0.0
    dq_l = dq[0]

    for i in range(1, plev):
        cur_age = tau[i]
        cur_dq = dq[i]
        cur_dq_l = cur_dq + dq_l

        if cur_dq < 0.0:
            cur_mu = (cur_age * cur_dq + dq_l * cur_mu) / cur_dq_l

        # update dq_l regardless
        dq_l = 0.0 if cur_dq_l > 0.0 else cur_dq_l

    return cur_mu


def age_precip_column(tau, dq):
    plev = tau.shape[1]
    cur_mu = 0.0
    dq_l = dq[:,0,:,:]

    for i in range(1, plev):
        cur_age = tau[:,i,:,:]
        cur_dq = dq[:,i,:,:]
        cur_dq_l = cur_dq + dq_l
        
        mask = cur_dq < 0.0
        
        cur_mu += mask * (cur_age * cur_dq + dq_l * cur_mu) / cur_dq_l

        # update dq_l regardless
        mask2 = cur_dq_l <0
        dq_l = mask2 * cur_dq_l

    return cur_mu